In [1]:
#Importing the Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt 


In [ ]:
# Reading customer dataset
customers = pd.read_csv('customer_data.csv.zip', compression='zip')
customers.info()
customers.describe()
print(customers.head())

In [ ]:
# Checking Null values in customer data

print(customers.isnull().sum())

In [ ]:
# Handling the null values in age column

customers['age'] = customers['age'].fillna(customers['age'].median())
print(customers.isnull().sum())

In [ ]:
# Checking the duplicates in customer data


customers_duplicates = customers.duplicated().sum()
print(customers_duplicates)

In [ ]:
# Sales dataset

sales = pd.read_csv('sales_data.csv.zip', compression='zip')
sales.info()
sales.describe()
print(sales.head())

In [ ]:
# Inspecting dataset

sales.info()
sales.describe()

In [ ]:
# Checking null values

sales.isnull().sum()

In [ ]:
# Checking the duplicates in sales Data

sales_duplicates = sales.duplicated().sum()
print(sales_duplicates)

In [10]:
# Converting Invoice date Datatype from string to Date

sales['invoice_date'] = pd.to_datetime(sales['invoice_date'], format='%d-%m-%Y')

In [ ]:
# Merging two dataset

customers_sales = customers.merge(sales, on = 'customer_id', how='inner')
customers_sales

In [ ]:
# ----Finding out the total Revenue
customers_sales['total_revenue'] = customers_sales['price'] * customers_sales['quantity']
print(f"Total Revenue: ${customers_sales['total_revenue'].sum():,.2f}")

In [ ]:
# Create age groups
customers_sales['age group'] = pd.cut(customers_sales['age'], bins=[0, 30, 45, 120],labels=['Young (18-30)', 'Middle (31-45)', 'Senior (46+)'])
customers_sales['age group']

In [ ]:
# Boxplot to check outliers in price, quantity and revenue
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
sns.boxplot(y=customers_sales['price'])
plt.title('Price Distribution')

plt.subplot(1, 3, 2)
sns.boxplot(y=customers_sales['quantity'])
plt.title('Quantity Distribution')

plt.subplot(1, 3, 3)
sns.boxplot(y=customers_sales['total_revenue'])
plt.title('Revenue Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot
Q1_price = customers_sales['price'].quantile(0.25)
Q3_price = customers_sales['price'].quantile(0.75)
IQR_price = Q3_price - Q1_price
price_outliers = customers_sales[(customers_sales['price'] < Q1_price - 1.5*IQR_price) | (customers_sales['price'] > Q3_price + 1.5*IQR_price)]
print(f"Number of price outliers: {len(price_outliers)} ({len(price_outliers)/len(customers_sales)*100:.2f}%)")

In [ ]:
# Only numeric columns for correlation
numeric_data = customers_sales[['age', 'quantity', 'price', 'total_revenue']]

# Calculate correlation
correlation = numeric_data.corr()

# Plot heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(correlation, annot=True, cmap='RdBu', center=0)
plt.title('Correlation Between Numeric Variables')
plt.show()

In [ ]:
# Age distribution ---->visuals

sns.histplot(customers_sales['age'], bins=25)
plt.xlabel('Age (years)')
plt.ylabel('Number of Customer')
plt.title('Age Distribution of Customers')
plt.show()

In [ ]:
# Age distribution ---> total spent by each group

sns.barplot(customers_sales, x = 'age group', y = 'total_revenue', estimator=sum)
plt.ylim(0, customers_sales['total_revenue'].sum())
plt.title('Total Revenue by Customer Age Group')
plt.xlabel('Age Group')
plt.ylabel('Total Revenue ($)')
plt.show()

In [ ]:
# Trend Analysis Visual----> invoice date by sales

# Replace your existing plot code with this
month = customers_sales['invoice_date'].dt.to_period('M')
monthly_revenue = customers_sales.groupby(month)['total_revenue'].sum()

plt.figure(figsize=(12,5))
plt.plot(monthly_revenue.index.astype(str), monthly_revenue.values, marker='o', linewidth=2)
plt.xlabel('Purchase Date', fontsize=12)
plt.ylabel('Total Revenue ($)', fontsize=12)
plt.title('Total Revenue over Time', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Insight 1: Age Analysis - Max spent age group.

# Grouping the ages in three categories
customers_sales['age group'] = pd.cut(customers_sales['age'], bins=[18,30,45,120], labels=['Young (18-30)', 'Middle (31-45)', 'Senior (46+)'])

# Performing groupby and aggration to get the total summation by age group
revenue_by_agegroup = customers_sales.groupby('age group')['total_revenue'].sum()

# getting the maximum revenue generated by age group
revenue_by_agegroup.idxmax()


In [ ]:
# Insight 2: maximum spent age group.

# getting the maximum revenue generated by age group
revenue_by_agegroup.idxmin()

In [ ]:
# Insight 3: Gender who visited mostly

most_visited_gender = customers_sales['gender'].value_counts().idxmax()
most_visited_gender

In [ ]:
# Insight 4: Gender with high revenue and low
revenue_by_gender = customers_sales.groupby('gender')['total_revenue'].sum()
print("Gender with high revenue: "+revenue_by_gender.idxmax())
print("Gender with low revenue: "+revenue_by_gender.idxmin())


In [ ]:
# Insight 5: Most preferred and least preffered payment method by customers
payment_counts = customers_sales['payment_method'].value_counts()
most_preferred = payment_counts.idxmax()
least_preferred = payment_counts.idxmin()
print(f"Most preferred payment method: {most_preferred} ({payment_counts[most_preferred]} transactions)")
print(f"Least preferred payment method: {least_preferred} ({payment_counts[least_preferred]} transactions)")

In [ ]:
# Insight 6: Category which generated high and low revenue

category_revenue = customers_sales.groupby(customers_sales['category'])['total_revenue'].sum()
print("Category which generated high revenue " + category_revenue.idxmax())
print("Category which generated low revenue " + category_revenue.idxmin())



In [ ]:
# Insight 7: Branch with highest and lowest revenue

shopping_mall_revenue = customers_sales.groupby(customers_sales['shopping_mall'])['total_revenue'].sum()
print("High revenue mall " + shopping_mall_revenue.idxmax())


In [ ]:
# Insight 8: Category that has sold high and low quantity

category_quantity = customers_sales.groupby(customers_sales['category'])['quantity'].sum()
print("highest sales volume " + category_quantity.idxmax())
print("low sales volume " +category_quantity.idxmin())


In [ ]:
# insight 9: Most loyal customer

customer_purchase_count = customers_sales.groupby('customer_id')['invoice_no'].nunique()
print("Customer with highest purchase is ", customer_purchase_count.idxmax())

In [ ]:
# insight 10: Best month for sales

customers_sales['month'] = customers_sales['invoice_date'].dt.month
sales_month = customers_sales.groupby('month')['total_revenue'].sum()
print("Month with highest revenue:", sales_month.idxmax())
print("Month with lowest revenue:", sales_month.idxmin())